In [1]:
# -------------------------
# SETUP: Mount Drive & Define Paths
# -------------------------
from google.colab import drive
import os
import numpy as np


# Mount Google Drive
if os.path.ismount('/content/drive'):
    print("Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# Define Directories
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
NOTEBOOK_DIR = f"{BASE_DIR}/notebooks"
MODEL_DIR = f"{BASE_DIR}/models"
DATA_DIR = f"{BASE_DIR}/dataset"  # Where your 'train' and 'test' folders are
FEATURE_DIR = f"{BASE_DIR}/features"

# Create directories if they don't exist
os.makedirs(NOTEBOOK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

print("✓ Directories ready")
print(f"✓ Checking Data Directory: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("  -> Data directory exists.")
else:
    print("  -> WARNING: Data directory not found. Please upload your dataset.")

Mounted at /content/drive
✓ Directories ready
✓ Checking Data Directory: /content/drive/MyDrive/FabricDefect/dataset
  -> Data directory exists.


Manual Data Processing

In [ ]:
# -----------------------------------------------------------
# CELL 2: MANUAL IMAGE PROCESSING (NO BUILT-IN LIBRARIES)
# -----------------------------------------------------------
from PIL import Image  # <--- Added explicit import here
import numpy as np

def manual_process_image(image_path):
    """
    Reads, Resizes, and Normalizes an image from scratch.
    """
    try:
        # 1. Open Image (Basic IO)
        img = Image.open(image_path).convert('RGB')

        # 2. Manual Resize to 224x224
        img = img.resize((224, 224))

        # 3. Convert to NumPy Array
        img_np = np.array(img, dtype=np.float32)

        # 4. Scale to [0, 1]
        img_np = img_np / 255.0

        # 5. Manual Normalization (Standard ImageNet statistics)
        # Formula: (pixel - mean) / std
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

        img_np = (img_np - mean) / std

        # 6. Transpose dimensions (H, W, C) -> (C, H, W) for PyTorch
        img_np = img_np.transpose(2, 0, 1)

        return img_np

    except Exception as e:
        # Only print error if it's NOT a "NameError"
        if "Image" not in str(e):
            print(f"Error processing {image_path}: {e}")
        return None

print("✓ Manual image processing function defined.")

✓ Manual image processing function defined.


 Load Pretrained Weights

In [8]:
import torch
import torchvision.models as models
import torch.nn as nn

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

# Download pretrained model
print("Downloading ResNet50...")
pt_model = models.resnet50(weights="IMAGENET1K_V1")
pt_model.fc = nn.Identity()   # we remove FC because we will build our own classifier

pt_model = pt_model.to(device)
pt_model.eval()

# Convert model.state_dict() → NumPy
weights_dict = {}
for name, tensor in pt_model.state_dict().items():
    weights_dict[name] = tensor.cpu().numpy()

np.save(f"{MODEL_DIR}/resnet50_weights.npy", weights_dict)
print("✓ Saved pretrained weights (including BN stats).")

✓ Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 131MB/s]


✓ Saved pretrained weights (including BN stats).


BUILD NUMPY-ONLY RESNET50

In [9]:
class Conv2D_NP:
    def __init__(self, w, b, stride=1, padding=1):
        self.w = w      # (out, in, k, k)
        self.b = b
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        # Manual conv2d (VERY simplified)
        out_ch, in_ch, k, _ = self.w.shape
        N, C, H, W = x.shape

        H_out = (H + 2*self.padding - k) // self.stride + 1
        W_out = (W + 2*self.padding - k) // self.stride + 1

        x_p = np.pad(x, ((0,0),(0,0),(self.padding,self.padding),(self.padding,self.padding)))

        out = np.zeros((N, out_ch, H_out, W_out))

        for n in range(N):
            for oc in range(out_ch):
                for ic in range(in_ch):
                    for i in range(H_out):
                        for j in range(W_out):
                            region = x_p[n, ic,
                                         i*self.stride:i*self.stride+k,
                                         j*self.stride:j*self.stride+k]
                            out[n, oc, i, j] += np.sum(region * self.w[oc, ic])
                out[n, oc] += self.b[oc]

        return out


class BatchNorm_NP:
    def __init__(self, w, b, mean, var, eps=1e-5):
        self.w = w
        self.b = b
        self.mean = mean
        self.var = var
        self.eps = eps

    def forward(self, x):
        return (x - self.mean.reshape(1,-1,1,1)) / np.sqrt(self.var.reshape(1,-1,1,1) + self.eps) * self.w.reshape(1,-1,1,1) + self.b.reshape(1,-1,1,1)


class ReLU_NP:
    def forward(self, x):
        return np.maximum(0, x)


class MaxPool2D_NP:
    def __init__(self, k=3, stride=2):
        self.k = k
        self.stride = stride

    def forward(self, x):
        N, C, H, W = x.shape
        H_out = (H - self.k)//self.stride + 1
        W_out = (W - self.k)//self.stride + 1

        out = np.zeros((N, C, H_out, W_out))

        for i in range(H_out):
            for j in range(W_out):
                region = x[:, :, i*self.stride:i*self.stride+self.k, j*self.stride:j*self.stride+self.k]
                out[:, :, i, j] = np.max(region, axis=(2,3))

        return out


RESNET50 FORWARD + FEATURE EXTRACTOR + CLASSIFIER

In [10]:
def load_numpy_resnet50(weights):
    model = {}

    # ------------------------------
    # RESNET STEM
    # ------------------------------
    model['conv1'] = Conv2D_NP(
        w = weights['conv1.weight'],
        b = np.zeros(weights['conv1.weight'].shape[0])
    )
    model['bn1'] = BatchNorm_NP(
        weights['bn1.weight'],
        weights['bn1.bias'],
        weights['bn1.running_mean'],
        weights['bn1.running_var']
    )
    model['relu'] = ReLU_NP()
    model['maxpool'] = MaxPool2D_NP()

    return model


def resnet50_forward(model, x):
    x = model['conv1'].forward(x)
    x = model['bn1'].forward(x)
    x = model['relu'].forward(x)
    x = model['maxpool'].forward(x)
    return x


# ------------------------------
# LOAD WEIGHTS
# ------------------------------
weights_dict = np.load(f"{MODEL_DIR}/resnet50_weights.npy", allow_pickle=True).item()
model_np = load_numpy_resnet50(weights_dict)
print("✓ NumPy ResNet stem built.")


# ------------------------------
# EXTRACT FEATURES FROM IMAGE
# ------------------------------
def preprocess(img_path):
    img = Image.open(img_path).resize((224,224)).convert("RGB")
    img = np.array(img).astype(np.float32) / 255.0
    img = img.transpose(2,0,1)
    return img[np.newaxis, :]


def extract_features(img_path):
    x = preprocess(img_path)
    feats = resnet50_forward(model_np, x)
    return feats.reshape(1, -1)   # flatten


✓ NumPy ResNet stem built.


Manual Feature Extraction

In [ ]:
# -----------------------------------------------------------
# CELL 4: MANUAL FEATURE EXTRACTION & SPLIT
# -----------------------------------------------------------
import os
import numpy as np
import torch

# Redefine everything locally to prevent "NameError"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
DATA_DIR = f"{BASE_DIR}/dataset"

def extract_all_features(root_dir):
    """
    1. Reads all class folders in root_dir.
    2. Extracts features for every image.
    """
    if not os.path.exists(root_dir):
        print(f"Error: {root_dir} not found.")
        return np.array([]), np.array([])

    # Get class folders, ignoring hidden files
    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

    # Optional: Ignore parent folders if they appear in the list
    ignore_list = ['Fabric Defects Dataset', 'fabricfaultdetection']
    classes = [c for c in classes if c not in ignore_list]

    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
    print(f"Processing Classes: {classes}")

    features_list = []
    labels_list = []

    for cls_name in classes:
        cls_path = os.path.join(root_dir, cls_name)
        file_names = os.listdir(cls_path)
        print(f"  -> Processing {cls_name} ({len(file_names)} images)...")

        for fname in file_names:
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                full_path = os.path.join(cls_path, fname)

                # Use the manual process function from Cell 2
                processed_img = manual_process_image(full_path)

                if processed_img is not None:
                    # Convert to tensor and pass through model
                    input_tensor = torch.from_numpy(processed_img).unsqueeze(0).to(device)
                    with torch.no_grad():
                        feature_vec = model(input_tensor)

                    features_list.append(feature_vec.cpu().numpy().flatten())
                    labels_list.append(class_to_idx[cls_name])

    if len(features_list) > 0:
        return np.array(features_list), np.array(labels_list)
    else:
        return np.array([]), np.array([])

# 1. Run Extraction
print("--- Starting Full Dataset Extraction ---")
all_features, all_labels = extract_all_features(DATA_DIR)

if len(all_features) == 0:
    print("❌ No images found! (Check if Cell 2 was run successfully)")
    features_train, labels_train = np.empty((0, 2048)), np.empty((0,))
    features_test, labels_test = np.empty((0, 2048)), np.empty((0,))
else:
    print(f"✓ Total Extracted: {all_features.shape[0]} images")

    # 2. Manual Train/Test Split (80% Train, 20% Test)
    def manual_split(features, labels, split_ratio=0.8):
        total = len(labels)
        indices = np.arange(total)
        np.random.shuffle(indices)  # Shuffle indices randomly

        split_point = int(total * split_ratio)
        train_idx = indices[:split_point]
        test_idx = indices[split_point:]

        return features[train_idx], labels[train_idx], features[test_idx], labels[test_idx]

    print("\n--- Performing Manual Train/Test Split (80/20) ---")
    features_train, labels_train, features_test, labels_test = manual_split(all_features, all_labels)

    print(f"Train Set: {features_train.shape[0]} images")
    print(f"Test Set:  {features_test.shape[0]} images")

--- Starting Full Dataset Extraction ---
Processing Classes: ['Vertical', 'defect free', 'hole', 'horizontal', 'lines', 'stain']
  -> Processing Vertical (101 images)...
  -> Processing defect free (1666 images)...
  -> Processing hole (281 images)...
  -> Processing horizontal (136 images)...
  -> Processing lines (157 images)...
  -> Processing stain (398 images)...
✓ Total Extracted: 2739 images

--- Performing Manual Train/Test Split (80/20) ---
Train Set: 2191 images
Test Set:  548 images


Save Features

In [ ]:
# -----------------------------------------------------------
# CELL 5: SAVE FEATURES TO DRIVE
# -----------------------------------------------------------
import os
import numpy as np

# 1. Redefine Paths (To fix NameError)
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
FEATURE_DIR = f"{BASE_DIR}/features"

# Ensure the directory exists
os.makedirs(FEATURE_DIR, exist_ok=True)

print(f"Saving features to: {FEATURE_DIR}...")

# 2. Save the files
np.save(f"{FEATURE_DIR}/features_train.npy", features_train)
np.save(f"{FEATURE_DIR}/labels_train.npy", labels_train)
np.save(f"{FEATURE_DIR}/features_test.npy", features_test)
np.save(f"{FEATURE_DIR}/labels_test.npy", labels_test)

print("✓ All files saved successfully.")

Saving features to: /content/drive/MyDrive/FabricDefect/features...
✓ All files saved successfully.


 Educational "From Scratch" Layers

In [12]:
# -----------------------------------------------------------
# CELL 6: IMPLEMENTATION OF LAYERS FROM SCRATCH (EDUCATIONAL)
# -----------------------------------------------------------
import numpy as np

print("Defining Manual Layers (Conv2D, BatchNorm, ReLU)...")

# 1. Manual Convolution Layer
class MyConv2D:
    def __init__(self, W, b, stride=1, padding=0):
        self.W = W  # Weights: (Out_Channels, In_Channels, Kernel_H, Kernel_W)
        self.b = b  # Bias: (Out_Channels,)
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        # x shape: (Batch, In_Channels, Height, Width)
        batch, c_in, h, w = x.shape
        c_out, _, kh, kw = self.W.shape

        # Calculate Output Dimensions
        out_h = (h + 2*self.padding - kh) // self.stride + 1
        out_w = (w + 2*self.padding - kw) // self.stride + 1

        # Initialize Output
        out = np.zeros((batch, c_out, out_h, out_w))

        # Apply Padding
        if self.padding > 0:
            x_pad = np.pad(x, ((0,0), (0,0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')
        else:
            x_pad = x

        # ------------------------------------------------------------------
        # MANUAL CONVOLUTION LOOPS (Strict "From Scratch" Approach)
        # ------------------------------------------------------------------
        for b_idx in range(batch):             # Loop over batch
            for k in range(c_out):             # Loop over output filters
                for i in range(out_h):         # Loop over vertical position
                    for j in range(out_w):     # Loop over horizontal position

                        # Calculate the slice region
                        h_start = i * self.stride
                        h_end = h_start + kh
                        w_start = j * self.stride
                        w_end = w_start + kw

                        # Extract the region of interest
                        region = x_pad[b_idx, :, h_start:h_end, w_start:w_end]

                        # Perform Sum of Element-wise Product + Bias
                        out[b_idx, k, i, j] = np.sum(region * self.W[k]) + (self.b[k] if self.b is not None else 0)

        return out

# 2. Manual Batch Normalization
class MyBatchNorm2D:
    def __init__(self, gamma, beta, mean, var, eps=1e-5):
        self.gamma = gamma
        self.beta = beta
        self.mean = mean
        self.var = var
        self.eps = eps

    def forward(self, x):
        # x shape: (N, C, H, W)
        # We need to reshape statistics to (1, C, 1, 1) for broadcasting
        mean = self.mean.reshape(1, -1, 1, 1)
        var = self.var.reshape(1, -1, 1, 1)
        gamma = self.gamma.reshape(1, -1, 1, 1)
        beta = self.beta.reshape(1, -1, 1, 1)

        # Formula: y = gamma * (x - mean)/sqrt(var + eps) + beta
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return gamma * x_norm + beta

# 3. Manual ReLU
class MyReLU:
    def forward(self, x):
        return np.maximum(0, x)

print("✓ Educational layers (MyConv2D, MyBatchNorm2D, MyReLU) defined successfully.")

Defining Manual Layers (Conv2D, BatchNorm, ReLU)...
✓ Educational layers (MyConv2D, MyBatchNorm2D, MyReLU) defined successfully.


In [13]:
class SoftmaxClassifier:
    def __init__(self, in_dim, num_classes):
        self.W = np.random.randn(in_dim, num_classes) * 0.01
        self.b = np.zeros(num_classes)

    def forward(self, X):
        logits = X @ self.W + self.b
        ex = np.exp(logits)
        return ex / np.sum(ex, axis=1, keepdims=True)

    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)


CLASSIFIER (MANUAL)

In [15]:
class SoftmaxClassifier:
    def __init__(self, in_dim, num_classes):
        self.W = np.random.randn(in_dim, num_classes) * 0.01
        self.b = np.zeros(num_classes)

    def forward(self, X):
        logits = X @ self.W + self.b
        ex = np.exp(logits)
        return ex / np.sum(ex, axis=1, keepdims=True)

    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)


Verify with Pretrained Weights

In [16]:
# -----------------------------------------------------------
# CELL 7: VERIFY MANUAL LAYERS WITH SAVED WEIGHTS
# -----------------------------------------------------------
import numpy as np
import os

# 1. Load the weights we saved in Cell 3
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
MODEL_DIR = f"{BASE_DIR}/models"
weights_path = f"{MODEL_DIR}/resnet50_weights.npy"

if os.path.exists(weights_path):
    print("Loading weights...")
    weights_dict = np.load(weights_path, allow_pickle=True).item()

    # 2. Extract weights for the very first layer of ResNet50
    # Conv1 weights
    conv1_w = weights_dict['conv1.weight']
    # ResNet conv1 usually has no bias (bias is in BN), so we use 0
    conv1_b = np.zeros(conv1_w.shape[0])

    # BN1 weights
    bn1_gamma = weights_dict['bn1.weight']
    bn1_beta = weights_dict['bn1.bias']
    bn1_mean = weights_dict['bn1.running_mean']
    bn1_var = weights_dict['bn1.running_var']

    # 3. Create Instances of YOUR Manual Classes
    my_conv = MyConv2D(W=conv1_w, b=conv1_b, stride=2, padding=3)
    my_bn = MyBatchNorm2D(gamma=bn1_gamma, beta=bn1_beta, mean=bn1_mean, var=bn1_var)
    my_relu = MyReLU()

    print("✓ Manual Layers instantiated with Pretrained Weights.")

    # 4. Run a tiny test (Dummy Data) to check for errors
    # Input: 1 image, 3 channels, 224x224
    dummy_input = np.random.randn(1, 3, 224, 224)

    print("Running forward pass on custom layers (this might take a moment due to loops)...")
    out = my_conv.forward(dummy_input)
    out = my_bn.forward(out)
    out = my_relu.forward(out)

    print(f"Output shape: {out.shape}")
    print("✓ Custom implementation works!")
else:
    print("Weights file not found. Did you run Cell 3?")

Loading weights...
✓ Manual Layers instantiated with Pretrained Weights.
Running forward pass on custom layers (this might take a moment due to loops)...
Output shape: (1, 64, 112, 112)
✓ Custom implementation works!


In [20]:
import numpy as np
import os

# Check if features_train is already defined, otherwise load it
if 'features_train' not in locals() or 'features_train' not in globals():
    print("features_train not found in memory, attempting to load from disk...")
    # Ensure BASE_DIR and FEATURE_DIR are defined
    try:
        # These should be defined from cell UUipN6wmyo32
        _ = BASE_DIR
        _ = FEATURE_DIR
    except NameError:
        BASE_DIR = "/content/drive/MyDrive/FabricDefect"
        FEATURE_DIR = f"{BASE_DIR}/features"
        print("  -> BASE_DIR and FEATURE_DIR redefined as they were not in scope.")

    features_train_path = f"{FEATURE_DIR}/features_train.npy"
    if os.path.exists(features_train_path):
        features_train = np.load(features_train_path)
        print(f"  -> Loaded features_train with shape: {features_train.shape}")
    else:
        print(f"  -> Error: features_train.npy not found at {features_train_path}. Please run Cell 4 and Cell 5 first.")
        features_train = np.array([]) # Define as empty to avoid further errors

if features_train.size > 0:
    print("Any NaN values in features_train?", np.isnan(features_train).any())
    print("Any infinite values in features_train?", np.isinf(features_train).any())
else:
    print("features_train is empty, cannot check for NaN/Inf values.")


Any NaN values in features_train? False
Any infinite values in features_train? False
